In [1]:
from pathlib import Path
from metasmith.agents import Agent
from metasmith.models.libraries import *
from metasmith.models.remote import *

from local.constants import WORKSPACE_ROOT
SOCKEYE_SOURCE = GlobusSource.Parse("https://app.globus.org/file-manager?origin_id=64a5c402-05c4-4607-bbad-46a9c2aebd98&origin_path=%2Fhome%2Ftxyliu%2F")
SOCKEYE_SOURCE.endpoint

'64a5c402-05c4-4607-bbad-46a9c2aebd98'

In [2]:
agent_local = Agent(
    home=Source.FromLocal(WORKSPACE_ROOT/"main/local_mock/cache/local_home"),
)

agent_ssh = Agent(
    home=SshSource(
        host="cosmos",
        path="~/workspace/metasmith_home",
    ).AsSource(),
)

agent_slurm = Agent(
    setup_commands=[
        "module load gcc/9.4.0 apptainer/1.3.1",
    ],
    home=SshSource(
        host="sockeye",
        path="~/scratch/metasmith_home",
    ).AsSource(),
    globus_uuid=SOCKEYE_SOURCE.endpoint,
)

# agent=agent_local
# agent=agent_ssh
agent=agent_slurm
agent.Deploy()

2025-03-14_13-49-56  | starting ssh to [sockeye]


2025-03-14_13-49-56 E| Pseudo-terminal will not be allocated because stdin is not a terminal.


2025-03-14_13-49-57  | ssh_connected_flag.1FQfCxG4
2025-03-14_13-49-58  | /scratch/st-shallam-1/pwy_group/metasmith_home
2025-03-14_13-49-58  | /arc/home/txyliu
2025-03-14_13-49-58  | >>> AGENT_HOME=/scratch/st-shallam-1/pwy_group/metasmith_home
2025-03-14_13-49-58  | >>> mkdir -p $AGENT_HOME
2025-03-14_13-49-58  | >>> mkdir -p /arc/home/txyliu/.globus
2025-03-14_13-49-58  | >>> mkdir -p /arc/home/txyliu/.globusonline
2025-03-14_13-49-58  | >>> [ -e /scratch/st-shallam-1/pwy_group/metasmith_home/metasmith.sif ] || apptainer pull /scratch/st-shallam-1/pwy_group/metasmith_home/metasmith.sif docker://quay.io/hallamlab/metasmith:latest
2025-03-14_13-49-58  | staged [msm_stub]
2025-03-14_13-49-58  | staged [msm]
2025-03-14_13-49-58  | >>> cd /scratch/st-shallam-1/pwy_group/metasmith_home && ./msm api deploy_from_container
2025-03-14_13-49-58  | including dev binds


2025-03-14_13-49-59 E| INFO:    squashfuse not found, will not be able to use gocryptfs
2025-03-14_13-49-59 E| INFO:    gocryptfs not found, will not be able to use gocryptfs


2025-03-14_13-50-06  | 2025-03-14_13-50-06  | api call to [deploy_from_container] with [{}]
2025-03-14_13-50-06  | 2025-03-14_13-50-06  | deploying to [/ws]
2025-03-14_13-50-06  | 2025-03-14_13-50-06  | deploying relay server to [/ws/relay/msm_relay]
2025-03-14_13-50-06  | 2025-03-14_13-50-06  | deployment complete
2025-03-14_13-50-07  | staged [lib/agent.yml]
2025-03-14_13-50-07  | staged [lib/msm_bootstrap]
2025-03-14_13-50-07  | staged [lib/nextflow_config]
2025-03-14_13-50-07  | deploying staged files
2025-03-14_13-50-09  | deployed to [ssh://sockeye:~/scratch/metasmith_home]


In [3]:
CACHE = WORKSPACE_ROOT/"main/local_mock/cache/xgdb_tests"
trlib = TransformInstanceLibrary.Load("./transforms/simple_1")
xgdb = DataInstanceLibrary.Load(CACHE/"test.xgdb")
# refdb = DataInstanceLibrary.Load(CACHE/"ref.xgdb")
types = DataTypeLibrary.Load(WORKSPACE_ROOT/"main/local_mock/prototypes/metagenomics.dev3.yml")

In [4]:
# chinook_ep = GlobusSource.Parse("https://app.globus.org/file-manager?origin_id=2602486c-1e0f-47a0-be15-eec1b0ff0f96&origin_path=%2FMetasmith%2F").endpoint
# refdb.SaveAs(GlobusSource(endpoint=chinook_ep, path="/Metasmith/ref.xgdb").AsSource())

refdb = DataInstanceLibrary.LoadFrom(
    src=GlobusSource.Parse("https://app.globus.org/file-manager?origin_id=2602486c-1e0f-47a0-be15-eec1b0ff0f96&origin_path=%2FMetasmith%2Fref.xgdb%2F").AsSource(),
    dest=CACHE/"ref.image.xgdb",
    as_image=True,
)
refdb.remote_src

Source(address='globus://2602486c-1e0f-47a0-be15-eec1b0ff0f96:/Metasmith/ref.xgdb', type=SourceType.GLOBUS)

In [5]:
task = agent.GenerateWorkflow(
    given=[xgdb, refdb],
    transforms=[trlib],
    targets=[
        types["orf_annotations"].WithLineage([
            types["contigs"],
            # xgdb["example.fna"].type,
        ]),
    ],
)

print(task.plan._key)
for step in task.plan.steps:
    print(step.transform.name)

aLR0oVmp
pprodigal
diamond


In [6]:
with open(WORKSPACE_ROOT/"secrets/slurm_account") as f:
    slurm_account = f.read().strip()
    
task.container_runtime = ContainerRuntime.APPTAINER
task.config = dict(
    nextflow = dict(
        preset = "slurm",
        slurm_account=slurm_account,
        # preset = "default",
    ),

)

In [7]:
agent.StageWorkflow(task, on_exist="clear")
# agent.StageWorkflow(task, on_exist="update")
# agent.StageWorkflow(task)

2025-03-14_13-50-09  | connecting to deployed agent
2025-03-14_13-50-09  | starting ssh to [sockeye]


 E| > Pseudo-terminal will not be allocated because stdin is not a terminal.


  | > ssh_connected_flag.1FQfCxG4
2025-03-14_13-50-10  | starting relay service
  | > 2025-03-14_13-50-11  | connecting to relay as [MGznC7StQLz6]


 E| > 2025-03-14_13-50-11 E| relay server already running in [relay/connections]
2025-03-14_13-50-12 W| task already staged at [~/scratch/metasmith_home/runs/aLR0oVmp]
2025-03-14_13-50-12 W| clearing previously staged task


2025-03-14_13-50-12  | sending metadata for workflow [aLR0oVmp]
2025-03-14_13-50-18  | staging
  | > including dev binds


 E| > INFO:    squashfuse not found, will not be able to use gocryptfs
 E| > INFO:    gocryptfs not found, will not be able to use gocryptfs


  | > 2025-03-14_13-50-18  | api call to [stage_workflow] with [{'task_key': 'aLR0oVmp'}]
  | > 2025-03-14_13-50-18  | staging workflow [aLR0oVmp] with [2] data libs and [1] transform libs
  | > 2025-03-14_13-50-19  | ex| /scratch/st-shallam-1/pwy_group/metasmith_home
  | > 2025-03-14_13-50-19  | work [/ws/runs/aLR0oVmp]
  | > 2025-03-14_13-50-19  | data [/msm_home/data]
  | > 2025-03-14_13-50-19  | external work [/scratch/st-shallam-1/pwy_group/metasmith_home/runs/aLR0oVmp]
  | > 2025-03-14_13-50-19  | external data [/scratch/st-shallam-1/pwy_group/metasmith_home/data]
  | > 2025-03-14_13-50-19  | additional params:
  | > 2025-03-14_13-50-19  |     nextflow:
  | > 2025-03-14_13-50-19  |       preset: slurm
  | > 2025-03-14_13-50-19  |       slurm_account: st-shallam-1
  | > 2025-03-14_13-50-19  | moving remote data libraries to [/msm_home/data]
  | > 2025-03-14_13-50-19  | using nextflow preset [slurm]
  | > 2025-03-14_13-50-19  | [aLR0oVmp] staged to [{AGENT_HOME}/runs/aLR0oVmp]
2025

In [8]:
# import shutil
# work_root = WORKSPACE_ROOT/"main/local_mock/cache/local_home/runs/kCvaS6w9"
# for p in [".nextflow", "nxf_logs", "nxf_work", "results"]:
#     shutil.rmtree(work_root/p, ignore_errors=True)
# shutil.rmtree(WORKSPACE_ROOT/"main/local_mock/mock/cache", ignore_errors=True)
    
agent.RunWorkflow(task)

2025-03-14_13-50-39  | connecting to deployed agent
2025-03-14_13-50-39  | starting ssh to [sockeye]


 E| > Pseudo-terminal will not be allocated because stdin is not a terminal.


  | > ssh_connected_flag.1FQfCxG4
2025-03-14_13-50-40  | starting relay service
  | > 2025-03-14_13-50-40  | connecting to relay as [LSo4BbGCQU7f]


 E| > 2025-03-14_13-50-40 E| relay server already running in [relay/connections]


2025-03-14_13-50-41  | executing workflow
  | > including dev binds


 E| > INFO:    squashfuse not found, will not be able to use gocryptfs
 E| > INFO:    gocryptfs not found, will not be able to use gocryptfs


  | > 2025-03-14_13-50-42  | api call to [execute_workflow] with [{'key': 'aLR0oVmp'}]
  | > 2025-03-14_13-50-42  | workspace [/msm_home/runs/aLR0oVmp]
  | > 2025-03-14_13-50-42  | external workspace [/scratch/st-shallam-1/pwy_group/metasmith_home/runs/aLR0oVmp]
  | > 2025-03-14_13-50-42  | executing workflow [aLR0oVmp] with preset [slurm]
  | > 2025-03-14_13-50-42  | preset [slurm]
  | > 2025-03-14_13-50-42  | steps [2]
  | > 2025-03-14_13-50-42  | locating input data with ag`ent's globus endpoint [64a5c402-05c4-4607-bbad-46a9c2aebd98]
  | > 2025-03-14_13-50-42  | [3RJW32jRo2ju] is at [/msm_home/runs/aLR0oVmp/_metasmith/task/transforms/3RJW32jRo2ju]
  | > 2025-03-14_13-50-42  | [gzLTT7PL67JN] is at [/msm_home/runs/aLR0oVmp/_metasmith/task/data/gzLTT7PL67JN]
  | > 2025-03-14_13-50-42  | [1AvO1xrdW7LS] is at [/msm_home/data/1AvO1xrdW7LS]
  | > 2025-03-14_13-50-42  | calling nextflow from container


 E| > 2025-03-14_13-50-42 E| Illegal option --


  | > 2025-03-14_13-50-50  |  N E X T F L O W   ~  version 24.10.5
  | > 2025-03-14_13-50-50  | 
  | > 2025-03-14_13-50-52  | WARN: It appears you have never run this project before -- Option `-resume` is ignored
  | > 2025-03-14_13-50-52  | Launching `./workflow.nf` [sick_agnesi] DSL2 - revision: 1b95ba5e23
  | > 2025-03-14_13-50-52  | 
  | > 2025-03-14_13-50-54  | [-        ] pprodigal__zgz5x1IB -
  | > 2025-03-14_13-50-54  | [-        ] diamond__lnNJuDqG   -
  | > 2025-03-14_13-50-54  | 
  | > 2025-03-14_13-50-54  | [-        ] pprodigal__zgz5x1IB | 0 of 1
  | > 2025-03-14_13-50-54  | [-        ] diamond__lnNJuDqG   -
  | > 2025-03-14_13-50-56  | 
  | > 2025-03-14_13-50-56  | executor >  slurm (1)
  | > 2025-03-14_13-50-56  | [f5/c500be] pprodigal__zgz5x1IB (1) | 0 of 1
  | > 2025-03-14_13-50-56  | [-        ] diamond__lnNJuDqG       -
  | > 2025-03-14_13-51-34  | 
  | > 2025-03-14_13-51-34  | executor >  slurm (1)
  | > 2025-03-14_13-51-34  | [f5/c500be] pprodigal__zgz5x1IB (1) | 0

In [9]:
# import mimetypes

# def istext(filename):
#     s=open(filename, encoding="latin1").read(512)
#     text_characters = "".join([chr(x) for x in range(32, 127)] + list("\n\r\t\b"))
#     translation_table = str.maketrans("", "", text_characters)
#     if not s:
#         # Empty files are considered text
#         return True
#     if "\0" in s:
#         # Files with null bytes are likely binary
#         return False
#     # Get the non-text characters (maps a character to itself then
#     # use the 'remove' option to get rid of the text characters.)
#     t = s.translate(translation_table)
#     # If more than 30% non-text characters, then
#     # this is considered a binary file
#     if float(len(t))/float(len(s)) > 0.30:
#         return False
#     return True

# # istext("/home/tony/workspace/tools/Metasmith/metasmith.sif")
# istext("/home/tony/workspace/tools/Metasmith/main/local_mock/cache/local_home/runs/dwfuH8Cz/nxf_work/dd/6d7e3eef979ec8010613bb376b63b6/container.diamond.oci.uri")

In [10]:
# from metasmith.coms.containers import ContainerRuntime

# s = ContainerRuntime.APPTAINER.name
# ContainerRuntime[s], s